# Gemma 4 31B (w4a16) en vLLM, como en Kaggle, servido para la sesión de Claude

Antes de ejecutar:
1. **Entorno de ejecución → Cambiar tipo de entorno → G4 GPU** (RTX PRO 6000 Blackwell, 96 GB: la más rápida aquí; si vLLM se cuelga al arrancar, el notebook reintenta solo en modo eager) o **A100**.
2. **Secretos** (icono de llave, a la izquierda): `KAGGLE_USERNAME` y `KAGGLE_KEY` (de kaggle.com → Settings → API → Create New Token), con acceso al notebook activado.
3. Haber aceptado la licencia de Gemma 4 en Kaggle (página del modelo `google/gemma-4`).

Luego **Entorno de ejecución → Ejecutar todo**. Al final se imprimen la URL y la clave: pásalas a la sesión de Claude. Deja la pestaña abierta (Colab desconecta por inactividad).

In [ ]:
# 1. La GPU
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'GPU: {torch.cuda.get_device_name(0)}, {GPU_GB:.0f} GB')
assert GPU_GB >= 22, 'Hace falta una L4 (24 GB) o una A100: la T4 no alcanza para el 31B.'

In [ ]:
# 2. Las versiones de la competencia (el wheelhouse de Kaggle: vllm 0.19.1, transformers 5.13.1, compressed-tensors 0.15.0.1)
!pip install -q vllm==0.19.1 transformers==5.13.1 compressed-tensors==0.15.0.1 kagglehub 2>&1 | tail -3
import importlib.metadata as md
for p in ('vllm', 'transformers', 'compressed-tensors', 'torch'):
    print(p, md.version(p))

In [ ]:
# 3. El modelo, desde Kaggle Models (el mismo que monta la competencia)
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
import kagglehub
MODEL_PATH = kagglehub.model_download('google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
print(MODEL_PATH)
print(subprocess.run(['du', '-sh', MODEL_PATH], capture_output=True, text=True).stdout)

In [ ]:
# 4. vLLM con las opciones de la competencia (tools/kaggle_eval.py): parsers gemma4, bfloat16, nombre del modelo
#    de la competencia. El contexto: 32.768 si cabe; si no, se reintenta con menos (una L4 sola).
import secrets, time, urllib.request, json
API_KEY = secrets.token_urlsafe(24)
SERVED = 'gemma-4-31b-it-qat-w4a16-ct'

def start_vllm(max_len, eager=False):
    log = open('/content/vllm.log', 'w')
    cmd = ['vllm', 'serve', MODEL_PATH, '--served-model-name', SERVED, '--host', '127.0.0.1', '--port', '8000',
           '--api-key', API_KEY, '--dtype', 'bfloat16', '--max-model-len', str(max_len),
           '--gpu-memory-utilization', '0.92', '--enable-auto-tool-choice', '--tool-call-parser', 'gemma4',
           '--reasoning-parser', 'gemma4'] + (['--enforce-eager'] if eager else [])
    env = dict(os.environ, VLLM_NO_USAGE_STATS='1', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)

def wait_ready(proc, timeout=1500):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            return False
        try:
            urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5)
            return True
        except Exception:
            time.sleep(10)
    return False

# Sin CUDA graphs (--enforce-eager) solo si el arranque normal falla o se cuelga: en Blackwell (la G4) hay reportes de
# cuelgues al capturar los grafos; eager es algo más lento pero arranca.
MAX_LEN, EAGER = None, False
for max_len in (32768, 24576, 16384, 12288):
    for eager in (False, True):
        print(f'starting vLLM with max_model_len={max_len}' + (' --enforce-eager' if eager else '') + ' ...')
        proc = start_vllm(max_len, eager)
        if wait_ready(proc, timeout=1500 if not eager else 1200):
            MAX_LEN, EAGER = max_len, eager
            break
        proc.kill(); proc.wait()
        log = open('/content/vllm.log').read()
        print(log[-1500:])
        if 'KV cache' in log or 'max_model_len' in log.split('Error')[-1]:
            break          # no cabe ese contexto: probar uno menor, no eager
    if MAX_LEN:
        break
assert MAX_LEN, 'vLLM no arrancó: mira /content/vllm.log'
print(f'vLLM listo, max_model_len={MAX_LEN}' + (' (eager)' if EAGER else ''))

In [ ]:
# 5. Prueba local: una llamada con herramienta, como las del agente
req = urllib.request.Request('http://127.0.0.1:8000/v1/chat/completions', method='POST',
    headers={'content-type': 'application/json', 'authorization': f'Bearer {API_KEY}'},
    data=json.dumps({'model': SERVED, 'max_tokens': 200, 'messages': [{'role': 'user', 'content': 'Call ping with x=1.'}],
                     'tools': [{'type': 'function', 'function': {'name': 'ping', 'parameters': {'type': 'object',
                               'properties': {'x': {'type': 'integer'}}}}}]}).encode())
t0 = time.time()
ans = json.load(urllib.request.urlopen(req, timeout=300))
print(f'{time.time() - t0:.1f} s', json.dumps(ans['choices'][0]['message'])[:500])

In [ ]:
# 6. El túnel (Cloudflare, sin cuenta) y los datos para la sesión de Claude
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
import re
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
                       stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        URL = m.group(0); break
    time.sleep(2)
assert URL, open('/content/tunnel.log').read()[-1500:]
print('Pasa esto a la sesión de Claude:')
print(f'  URL:   {URL}/v1')
print(f'  CLAVE: {API_KEY}')
print(f'  MODELO: {SERVED}   max_model_len={MAX_LEN}')

In [ ]:
# 7. Mantener viva la sesión y ver el uso (deja esta celda corriendo; detenla para parar)
import collections
while True:
    alive = proc.poll() is None and tun.poll() is None
    lines = [l for l in open('/content/vllm.log').read().splitlines() if 'throughput' in l or 'Error' in l]
    print(time.strftime('%H:%M:%S'), 'vLLM y túnel vivos' if alive else 'ALGO SE CAYÓ: revisa los logs', '|',
          (lines[-1][-160:] if lines else ''))
    if not alive:
        break
    time.sleep(60)